# 05b. Multi-Agent 선택 기준 ②: 수치 읽기, Fast/Slow Path, 같은 조건 비교표, 선택 기록

앞서 05a에서는 Single 에이전트의 실패 원인을 분류하고 먼저 개선한 뒤, 작업을 나누는 이득과 비용, Persona와 Context 경계, 그리고 Skill과 SubAgent의 차이를 살펴보았습니다. 이번 노트북 **05b**에서는 측정된 수치들을 올바르게 해석하고(5절), 단순한 요청을 짧은 경로로 처리하는 Fast/Slow Path를 확인하며(6절), 여러 구조를 동일한 조건의 표 하나로 모아 비교한 뒤(7절), 사례 유형별로 특정 구조를 선택한 이유를 기록으로 남깁니다(8절).

이 노트북은 앞선 노트북의 커널(kernel) 상태에 의존하지 않고 단독으로 실행할 수 있도록 구성되어 있습니다. 따라서 5~7절에서 활용하는 05a의 주요 실행 결과 몇 가지를 **준비 절**에서 먼저 다시 실행합니다.

## 학습 목표

- 수치의 비교 조건을 정확히 읽고, 전체 지연을 최장 실행 경로 기준으로 분해합니다.
- Fast/Slow Path와 공통 게이트를 설계하고, 라우팅 오판이 발생했을 때의 경로와 비용을 확인합니다.
- 동일한 조건의 비교표를 구성하고, 실험 결과로부터 주장할 수 있는 것과 없는 것을 구분합니다.
- 다섯 가지 구조가 동일한 하네스(예산, 도구 허용 목록, 응답 형식, 쓰기 게이트)를 통과하기 때문에 오직 구조적 차이만 공정하게 비교된다는 점을 코드로 확인합니다.
- 특정 구조를 선택한 이유, 포기한 요소, 그리고 결정을 재검토할 조건을 선택 기록으로 남깁니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 5 | 수치 읽기와 지연 분해 | 5 |
| 6 | Fast/Slow Path | 6 |
| 7 | 같은 조건 비교표 | 10 |
| 8 | 선택 기록과 선택 순서도 | 4 |
| 9 | 직접 해보기와 정리 | 5 |

## 호출 규모

**이 과정에서 모델 호출 규모가 가장 큰 노트북입니다.** 동일한 사례를 여러 구조에서 반복 실행하여 비교하기 때문입니다. 아래 표는 실행 전 예고 상한이며, 각 유료 셀은 실행 직전에 예상 호출 규모를 한 줄로 안내합니다. 모든 호출은 수강생 본인의 크레딧을 사용합니다.

| 범위 | 모델 호출(예고 상한) | EXA |
|---|---:|---:|
| 준비 절: 05a의 실행 세 개를 다시 실행(Single 기준선 4사례 20, Supervisor 20, `single_skills` 12) | 52 | 0회 |
| 6절 hybrid 단순·복합 사례, Fast Path 강제 | 30 + 24 | 0회 |
| 7절 같은 조건 비교표, 예산을 낮춘 실패 행 | 100 + 3 | 0회 |
| 기본 실행 합계 | 예고 상한 209 | 0회 |
| `RUN_OPTIONAL = True` (첫 코드 셀, 4유형 10사례 × 5구조 × 1회) | 예고 상한 330 더 | 0회 |
| 직접 해보기의 `RUN_EXERCISE = True` | 해보기 1이 15 | 0회 |

05a를 방금 실행했더라도 단독 실행 원칙을 지키기 위해 준비 절은 다시 실행되며, 그만큼 호출 수가 늘어납니다. `RUN_OPTIONAL`과 `RUN_EXERCISE`는 기본값이 `False`이므로 기본 전체 실행에는 추가 비용이 발생하지 않습니다. 실제 호출 수는 모델의 판단에 따라 실행할 때마다 조금씩 달라질 수 있습니다.

아래 코드는 실습에 필요한 패키지와 모듈을 불러오고, API 키 설정과 평가용 사례 데이터를 준비합니다.

In [ ]:
import inspect
import json
import re
import time
from collections import Counter, defaultdict
from uuid import uuid4

from deepagents import create_deep_agent

from mas_lab.contracts import FinalAnswer, Proposal, validate_final
from mas_lab.evaluation import evaluate_row, load_gold, summarize_rows
from mas_lab.harness import task_targets
from mas_lab.hybrid import FAST_TOOLS, build_hybrid
from mas_lab.ledger import check_refund
from mas_lab import runner
from mas_lab.runner import FIXED_BUDGET, STRUCTURES, build_retriever, make_desk, run_case, run_grid
from mas_lab.structures import OWNER_TOOLS, build_handoff, build_single, build_supervisor, gate_answer
from mas_lab.trace import ModelMeter, longest_path, timeline
from mas_lab.view import bars, message_table, read_design_log, show_table, write_design_log
from mas_lab.workers import ALL_TOOLS, ROLE_TOOLS, SKILL_TOOLS
from rag_lab.corpus import load_cases
from rag_lab.evidence import BudgetExceeded
from rag_lab.settings import ROOT, announce, key_status, make_model, require_key

# API 키 설정 여부를 확인합니다 (키가 없으면 여기서 실행을 멈추고 .env 설정 방법을 안내합니다).
settings = require_key()
print(key_status(settings))

# LLM 모델 객체를 초기화합니다 (객체만 생성하며 실제 API 호출은 발생하지 않습니다).
model = make_model()

# 실습용 사례 데이터와 사후 평가용 정답 데이터를 로드합니다.
cases = load_cases()
case_by_id = {case["id"]: case for case in cases}
gold = load_gold()  # 사후 평가에만 사용하며, 실행 경로의 프롬프트나 도구에는 전달되지 않습니다.

# 선택 실습 스위치 (True로 변경하면 7절의 전체 비교 실험인 모델 약 220~330회, 10~15분 규모를 실행합니다).
RUN_OPTIONAL = False
print(f"사례 {len(cases)}개, 유형 {sorted({case['type'] for case in cases})}, RUN_OPTIONAL={RUN_OPTIONAL}")

## 05a에서 다시 실행하는 값

5절 이후의 분석에서는 05a에서 생성했던 몇 가지 실행 결과를 사용합니다. 이 노트북은 05a의 커널 상태 없이도 독립적으로 동작해야 하므로, 필요한 값들만 아래 셀들에서 다시 실행하여 준비합니다. 코드의 내용은 05a의 해당 셀과 동일하며 새로운 개념이 추가되지는 않습니다.

| 값 | 05a에서의 위치 | 이 노트북에서 활용하는 곳 |
|---|---|---|
| `single_rows` | 1절 Single 기준선 4사례 | 5절 수치 읽기, 7절 같은 구조로 두 번 측정한 비교표 |
| `supervisor_row`, `double_pay` | 2절 Supervisor 동일 사례 | 5절 지연 분해, 6절 게이트 검증 표, 7절 두 번 측정한 비교표 |
| `skills_row` | 4절 `single_skills` 실행 | 5절 수치 읽기 |
| `role_of_call` | 3절 역할 판별 함수 | 5절 지연 분해, 6절 역할별 호출 집계 |

아래 코드는 Single 에이전트 구조로 기준선 4개 사례를 실행하고 결과 표를 출력합니다.

In [ ]:
# Single 에이전트 구조로 기준선 4개 사례를 실행합니다.
announce("Single 기준선 4사례 실행", model_calls=20)
BASELINE_IDS = ["complex-double-pay", "insufficient-unknown-payment", "scope-beta-from-alpha", "complex-ops-log"]
single_rows = [run_case("single", case_by_id[case_id], model=model) for case_id in BASELINE_IDS]

# 주요 실행 지표 컬럼을 지정하여 결과 표를 출력합니다.
RUN_COLUMNS = ["structure", "case_id", "status", "outcome", "model_calls", "tool_calls", "input_tokens", "output_tokens", "elapsed_seconds"]
show_table(single_rows, columns=RUN_COLUMNS)

In [ ]:
def run_custom(label, case, build):
    """run_case와 동일한 예산, 게이트, 검증 로직을 사용하되 에이전트 조립 방식만 바꿉니다. 원문 메시지까지 반환하며 실패도 행으로 기록합니다."""
    run_id, desk, meter = uuid4().hex, make_desk(case, label=label), ModelMeter()
    row = {"run_id": run_id, "structure": label, "case_id": case["id"], "type": case["type"], "repeat": 1, "status": "ok", "outcome": None,
           "errors": [], "answer": "", "uncertainty": [], "evidence_refs": [], "raw_messages": []}
    started = time.perf_counter()
    try:
        # 전달받은 build 함수로 에이전트를 생성하고 질문을 입력하여 실행합니다.
        state = build(desk).invoke({"messages": [{"role": "user", "content": case["question"]}]}, config={"callbacks": [meter], "recursion_limit": 60})
        row["raw_messages"], final = state["messages"], state.get("structured_response")
        if final is None:
            row.update(status="validation_failed", errors=["최종 응답이 없습니다."])
        else:
            # 쓰기 게이트와 최종 응답 형식 검증을 순서대로 통과시킵니다.
            final = gate_answer(final, desk, request_id=run_id, question=case["question"])
            problems = validate_final(final, desk.known_refs(), desk.source_text())
            row.update(status="validation_failed" if problems else "ok", errors=problems, outcome=final.outcome, answer=final.text,
                       evidence_refs=final.evidence_refs, uncertainty=final.uncertainty)
    except BudgetExceeded as error:
        row.update(status="budget_exceeded", errors=[str(error)])
    except Exception as error:  # 예외가 발생해도 노트북을 멈추지 않고 실패 상태로 기록합니다.
        row.update(status="model_error", errors=[f"{type(error).__name__}: {error}"[:300]])

    # 모델 호출 횟수, 토큰 사용량, 도구 호출 내역 등을 집계하여 행에 업데이트합니다.
    totals, report = meter.totals(), desk.budget.report()
    row.update(model_calls=totals["model_calls"], tool_calls=report["counts"].get("tool", 0), input_tokens=totals["input_tokens"],
               output_tokens=totals["output_tokens"], usage_missing=totals["usage_missing"], elapsed_seconds=round(time.perf_counter() - started, 3),
               ledger_refunds=len(desk.ledger.refunds), calls=meter.calls, spans=desk.trace.spans, tools_called=[e["name"] for e in report["events"] if e["event"] == "tool"],
               denied=[e["name"] for e in report["events"] if e["event"] == "tool_denied"], retrieved_docs=sorted({e.doc_id for e in desk.context.evidence.values()}),
               cited_docs=sorted({e.doc_id for ref, e in desk.context.evidence.items() if ref in row["evidence_refs"]}))
    return row

print("run_custom", inspect.signature(run_custom), "| 고정 예산:", FIXED_BUDGET, "| 모델 호출 0회")

In [ ]:
# Supervisor 구조로 복합 사례(complex-double-pay)를 실행하고 메시지 흐름을 확인합니다.
announce("Supervisor로 동일 복합 사례 실행", model_calls=20)
double_pay = case_by_id["complex-double-pay"]
supervisor_row = run_custom("supervisor", double_pay, lambda desk: build_supervisor(desk, model))
show_table([supervisor_row], columns=RUN_COLUMNS)
show_table(message_table(supervisor_row["raw_messages"]))

In [ ]:
# Single+Skills(single_skills) 구조로 동일한 복합 사례를 실행합니다.
announce("single_skills로 동일 복합 사례 실행", model_calls=12)
skills_row = run_case("single_skills", double_pay, model=model)
show_table([skills_row], columns=RUN_COLUMNS)
print("읽은 Skill:", skills_row["loaded_skills"] or "없음")
print("호출한 도구:", dict(Counter(skills_row["tools_called"])))

In [ ]:
def role_of_call(call):
    """모델 호출 시 노출된 도구 목록을 기준으로 해당 호출의 역할을 판별합니다 (05a 3절과 동일한 함수입니다)."""
    tools = set(call.get("tools") or [])
    if "task" in tools:
        return "supervisor"
    if not tools:
        return "router"
    matched = [role for role, names in ROLE_TOOLS.items() if tools <= names]
    return matched[0] if matched else ("fast" if tools == ROLE_TOOLS["policy"] | ROLE_TOOLS["billing"] else "전체 도구")

print("준비 완료 / 모델 호출 0회")

**관찰할 것.** 위의 세 실행 셀은 05a의 동일한 실험을 다시 실행한 결과입니다. 앞서 05a를 실행해 보셨다면 동일한 사례의 `호출`, `입력 토큰`, `시간`, `자동 검사` 수치가 05a의 표와 일치하는지 비교해 보세요. 동일한 구조, 동일한 사례, 동일한 조건임에도 불구하고 실행할 때마다 수치가 조금씩 달라질 수 있습니다. 이러한 자연 변동폭이 7절에서 다룰 "같은 구조로 두 번 측정한 비교표"를 해석하는 중요한 기준선이 됩니다.

## 5. 수치 읽기와 지연 분해
**문제.** "토큰을 3배 더 쓴다"거나 "2배 빠르다"와 같은 문장은 비교 조건이 함께 제시되지 않으면 아무런 의미가 없습니다. 무엇과 무엇을 비교했는지(비교 대상), 어떤 지표로 측정했는지(평가 방법), 몇 번 반복해 측정했는지(표본), 그리고 누가 측정했는지(측정자) 중 하나라도 빠져 있다면 타당한 근거로 주장할 수 없습니다. 아래 셀은 외부의 임의 수치를 가져오지 않고, **방금 이 노트북에서 직접 실행해 얻은 표의 수치만** 사용하여 비교 문장을 만들고 네 가지 조건 칸으로 나누어 점검합니다.

지연 시간(Latency)의 해석도 마찬가지입니다. Worker를 세 개로 나누었다고 해서 처리 속도가 세 배 빨라진다는 생각은 흔한 오해입니다. 전체 지연 시간은 에이전트의 개수가 아니라 **가장 긴 실행 경로(Critical Path)**에 의해 결정되며, 그 경로에는 작업을 나누고 결과를 합치는 조정·집계 시간이 추가로 포함됩니다. `supervisor` 실행 행의 `spans`에는 각 Worker의 실행 구간만 기록되고 Supervisor 자체의 조정 구간은 빠져 있으므로, 두 구간을 함께 살펴보기 위해 모델 호출 기록(`calls`)의 시작과 종료 시각으로 전체 타임라인 구간을 재구성합니다.

아래 코드는 방금 측정한 수치들로 만든 네 가지 문장이 타당한 비교 조건을 갖추고 있는지 판정합니다.

In [ ]:
def times(a, b):
    """두 수치의 비율을 소수점 둘째 자리까지 계산합니다."""
    return round(a / b, 2) if a and b else "값 없음"

# 방금 실행한 Single과 Supervisor, Single+Skills의 수치를 가져와 비교 문장을 구성합니다.
single_double = single_rows[0]
mean_single = sum(row["elapsed_seconds"] for row in single_rows) / len(single_rows)
BLANK_FIELDS = ["비교 대상", "평가 방법", "표본", "측정자"]
claims = [
    {"문장": f"Supervisor는 Single의 {times(supervisor_row['input_tokens'], single_double['input_tokens'])}배 입력 토큰을 썼다", "비교 대상": "complex-double-pay에서 single과 supervisor", "평가 방법": "ModelMeter 입력 토큰 합계", "표본": "각 1회", "측정자": "내 노트북 실행"},
    {"문장": f"Single+Skills는 Single의 {times(skills_row['elapsed_seconds'], single_double['elapsed_seconds'])}배 시간이 걸렸다", "비교 대상": "complex-double-pay에서 single과 single_skills", "평가 방법": "run_case의 elapsed_seconds", "표본": "각 1회", "측정자": "내 노트북 실행"},
    {"문장": f"Supervisor({supervisor_row['elapsed_seconds']:.0f}초)와 Single 평균({mean_single:.0f}초)을 비교하면 시간이 {times(supervisor_row['elapsed_seconds'], mean_single)}배다", "비교 대상": "", "평가 방법": "elapsed_seconds", "표본": "1회와 4사례 평균", "측정자": "내 노트북 실행"},
    {"문장": "Supervisor가 Single보다 답의 품질이 좋다", "비교 대상": "complex-double-pay에서 single과 supervisor", "평가 방법": "", "표본": "각 1회", "측정자": ""}]

def verdict(claim):
    """비교 조건 중 빈 칸이 있는지 확인하여 주장의 타당성을 판정합니다."""
    blanks = [field for field in BLANK_FIELDS if not claim[field]]
    if blanks:
        return f"주장할 수 없음: {', '.join(blanks)} 빈 칸"
    return "1회 관찰로만 말할 수 있음" if "1회" in claim["표본"] else "조건을 확인한 뒤 말할 수 있음"

show_table([{**claim, "판정": verdict(claim)} for claim in claims])
print("셋째 문장은 사례 구성이 서로 달라 비교 대상이 성립하지 않으므로 칸을 비웠습니다. 모든 수치는 이 노트북의 실행 결과에서 가져왔습니다. 모델 호출 0회.")

In [ ]:
def call_spans(row):
    """개별 모델 호출 하나를 하나의 시간 구간(span)으로 변환합니다. 호출 사이의 빈 시간은 도구 실행과 코드 처리 시간입니다."""
    done = [call for call in row["calls"] if call.get("end")]
    origin = min((call["start"] for call in done), default=0.0)
    return [{"actor": f"{role_of_call(call)}#{index}", "kind": role_of_call(call), "start": round(call["start"] - origin, 3), "end": round(call["end"] - origin, 3)}
            for index, call in enumerate(done)]

# Supervisor 실행 기록에서 호출 구간과 선후 의존 관계를 추출하여 최장 경로를 계산합니다.
spans = call_spans(supervisor_row)
durations = {span["actor"]: span["end"] - span["start"] for span in spans}
depends = {span["actor"]: [other["actor"] for other in spans if other["start"] < span["start"] and other["end"] <= span["start"]] for span in spans}
serial = sum(durations.values())
critical = longest_path(durations, depends)

# Supervisor의 조정·집계 시간과 Worker들의 실행 시간을 구분하여 합산합니다.
by_kind = defaultdict(float)
for span in spans:
    by_kind["Supervisor 조정·집계" if span["kind"] == "supervisor" else "Worker 합계"] += span["end"] - span["start"]

print(timeline(spans))
print()
print(bars({"실제 경과(ms)": supervisor_row["elapsed_seconds"] * 1000, "모델 직렬 합(ms)": serial * 1000, "모델 최장 경로(ms)": critical * 1000,
            **{f"{name}(ms)": seconds * 1000 for name, seconds in by_kind.items()}}))
print(f"\nWorker 역할 {sum(kind in ROLE_TOOLS for kind in {span['kind'] for span in spans})}개, 직렬 합 / 최장 경로 = {times(serial, critical)}배, 실제 경과 - 최장 경로 = 도구와 코드 {supervisor_row['elapsed_seconds'] - critical:.2f}초. 모델 호출 0회.")

### 그림으로 보기: 지연은 어느 경로가 정하는가

위 셀에서 출력한 타임라인의 핵심 구조만 뽑아 다이어그램으로 나타냈습니다. `call_spans` 함수가 개별 모델 호출을 하나의 시간 구간으로 만들고, `longest_path` 함수가 전체 지연 시간을 결정하는 가장 긴 경로를 찾아냅니다.

![Supervisor의 위임 호출 뒤 billing, policy, outage Worker 호출이 겹쳐서 돌고 그 결과를 받아 Supervisor의 집계 호출이 이어진다. 경로는 위임 호출, 가장 느린 Worker, 집계 호출이다](./assets/05b-1-critical-path.png)

- 여러 Worker가 동시에 겹쳐서 실행되더라도 전체 경로에는 **가장 느린 Worker 하나**의 시간만 반영됩니다. 단순히 Worker 수를 세어 나눌 수 없습니다.
- 앞뒤에 위치한 두 번의 Supervisor 호출은 작업을 분리하면서 새롭게 추가된 조정 비용입니다. 이 시간이 Worker 병렬화로 아낀 시간보다 크다면, 작업을 나누기 전보다 전체 실행이 오히려 더 느려집니다.

**관찰할 것.** 첫 번째 표에서 `판정` 컬럼을 확인해 보세요. 계산된 숫자가 맞더라도 비교 조건에 빈 칸이 있으면 온전한 주장이 될 수 없습니다. 세 번째 문장은 나눗셈 계산 자체는 가능하지만 1개 사례와 4개 사례 평균을 비교했기 때문에 비교 대상이 성립하지 않습니다. 네 번째 문장은 구체적인 수치와 평가 방법, 측정자가 모두 빠져 있습니다. 여러분이 실험 결과를 보고 내리고 싶은 결론을 문장으로 추가해 동일한 네 칸에 채워 보세요. 외부 아티클이나 벤치마크에서 접하는 수치들도 이 네 가지 기준으로 나누어 보면 빠진 조건을 쉽게 찾을 수 있습니다.

두 번째 출력 결과는 에이전트 수가 아니라 **가장 긴 경로(Critical Path)** 관점으로 해석해야 합니다.

- 타임라인에서 세로 방향으로 겹쳐 보이는 막대가 동시(병렬) 실행 구간입니다. 만약 막대가 겹치지 않는다면, 코드로 강제한 `Send`가 아니라 모델의 자율 선택에 맡긴 병렬 호출이 이번 실행에서는 일어나지 않은 것입니다.
- `모델 직렬 합`과 `모델 최장 경로`의 차이가 병렬 실행을 통해 단축한 시간의 상한선입니다. 단축 비율이 Worker 수에 미치지 못한다면 무엇이 경로에 남아 있는지 확인해 보세요. 대개 가장 느린 Worker와 Supervisor의 호출 시간이 경로의 대부분을 차지합니다.
- `Supervisor 조정·집계` 시간은 에이전트를 나누면서 순수하게 추가된 비용입니다. 이 시간이 Worker 병렬화로 절약한 시간보다 크다면 Single 구조보다 더 느려집니다. 앞서 실행한 Single 행의 소요 시간과 함께 비교해 보세요.

그렇다면 모든 요청이 항상 이 무겁고 긴 경로를 거쳐야 할까요? 다음 절에서는 단순한 요청을 짧은 경로로 빠르게 처리하는 Fast Path를 살펴봅니다.

## 6. Fast/Slow Path
**문제.** 단순한 규정을 묻는 질문에 매번 세 개의 Worker를 모두 실행하는 것은 불필요한 비용 낭비입니다. 그래서 앞단에 Router를 두어 요청의 성격에 따라 경로를 나눕니다. 여기서 핵심은 **Fast Path가 안전 검사를 생략하는 지름길이 아니라는 점**입니다. Fast Path가 줄이는 것은 조사하는 도구와 에이전트의 폭이지, 최종 검증과 게이트가 아닙니다. 또한 Router는 언제든 오판할 수 있습니다. 복합 요청이 잘못해서 Fast Path로 들어갔을 때 어떤 일이 벌어지는지까지 확인해야 Fast Path의 진짜 이득을 평가할 수 있습니다.

`hybrid` 구조는 **범위 확인 → Router → Fast 또는 Worker 병렬 → 합성 → 검증 → 게이트** 순서로 동작하며, Fast 경로와 Slow 경로 모두 동일한 검증 노드와 쓰기 게이트를 통과합니다. 만약 Fast 경로에서 나온 응답의 `outcome`이 근거 부족(`insufficient`)이거나 형식 검증에서 문제가 발견되면 자동으로 Slow 경로로 전환(fallback)합니다.

**먼저 예측하세요.** 복합 사례인 `complex-double-pay`를 강제로 Fast Path에 보내면 어떤 일이 일어날까요? Fast 에이전트에게는 가용률 조회나 SLA 크레딧 계산 도구가 없습니다. 검증 단계에서 실패하여 Slow 경로로 전환될까요, 아니면 SLA 보상 부분만 누락된 채로 검증을 그대로 통과해 버릴까요? 또 근거가 부족한 `insufficient-unknown-payment` 사례는 어떻게 처리될까요? 실행하기 전에 두 사례의 결과를 각각 예측해 적어 보세요.

### 그림으로 보기: Fast와 Slow가 같은 검증을 지난다

`hybrid` 그래프의 실행 경로를 다이어그램으로 나타냈습니다. 이어지는 코드 셀들에서 이 경로를 따라 실제로 어떤 일이 일어나는지 확인합니다.

![router가 fast 또는 worker로 보내고, worker 결과는 join과 writer를 거쳐 validate로, fast의 답은 바로 validate로 모인 뒤 gate로 간다. Fast에서 온 답이 근거 부족이면 validate가 worker로 되돌려 Slow로 전환한다](./assets/05b-2-fast-slow.png)

- Fast 경로와 Slow 경로 모두 마지막에 동일한 `validate`와 `gate` 노드를 통과합니다. Fast Path가 생략하는 것은 조사의 범위이지 안전 검증이 아닙니다.
- 가운데 점선 화살표는 Fast 경로의 응답이 불충분할 때 Slow 경로(`worker`)로 되돌리는 전환(fallback) 간선입니다. 이 간선이 실행되면 Fast에서 이미 소비한 호출 비용에 더해 Slow에서 다시 실행하는 호출 비용까지 모두 지불하게 됩니다.
- 단, 이 전환 간선은 `validate` 노드가 응답이 부족하다고 감지할 때만 작동합니다. 만약 누락된 조사 범위를 형식 검증이 잡아내지 못하면 전환 없이 그대로 종료되며, 바로 그때가 가장 주의해야 할 위험한 상황입니다.

아래 코드는 `hybrid` 구조로 단순 사례와 복합 사례를 각각 실행하여 Router의 경로 선택과 실행 지표를 확인합니다.

In [ ]:
# hybrid 구조로 단순 사례(simple-deadline)와 복합 사례(complex-double-pay)를 실행합니다.
announce("hybrid 단순 사례와 복합 사례 실행", model_calls=30)
hybrid_rows = [run_case("hybrid", case_by_id[case_id], model=model) for case_id in ("simple-deadline", "complex-double-pay")]

def path_row(row, note=""):
    """실행 결과에서 라우팅 판단, Slow 전환 여부, 비용 및 자동 검사 결과를 추출하여 요약 행을 만듭니다."""
    decided = [event.get("choice") for event in row["events"] if event["kind"] == "decision"]
    check, roles = evaluate_row(row, gold), Counter(role_of_call(call) for call in row["calls"])
    return {"사례": row["case_id"], "조건": note or "Router 판단", "경로 판단": decided, "Slow 전환": any(event["kind"] == "fallback_to_slow" for event in row["events"]),
            "호출": row["model_calls"], "역할별 호출": dict(roles), "입력 토큰": row["input_tokens"], "시간": row["elapsed_seconds"], "상태": row["status"],
            "outcome": row["outcome"], "자동 검사": check["자동 검사"], "누락 사실": check["누락 사실"]}

show_table([path_row(row) for row in hybrid_rows])
print(timeline(hybrid_rows[1]["spans"]))

In [ ]:
# 실패 재현: Router의 판단을 무시하고 두 복합·불충분 요청을 강제로 Fast Path로 보냅니다.
announce("Fast Path 강제 2사례 실행", model_calls=24)
forced_rows = [run_case("hybrid", case_by_id[case_id], model=model, force_path="fast") for case_id in ("complex-double-pay", "insufficient-unknown-payment")]
show_table([path_row(row, "Fast 강제") for row in forced_rows] + [path_row(hybrid_rows[1])])
for row in forced_rows:
    print(row["case_id"], "사건 순서:", [event["kind"] for event in row["events"] if event["kind"] != "tool"])
    print("  답 앞부분:", row["answer"][:120].replace("\n", " "))

In [ ]:
# Fast Path에서 넘어온 쓰기(환불) 제안도 동일하게 게이트(gate_answer)에서 통제되는지 확인합니다.
beta_desk, alpha_desk = make_desk(case_by_id["approval-beta-small"], label="gate"), make_desk(double_pay, label="gate")
requests = [(alpha_desk, "P-8101", 1_000_000), (alpha_desk, "P-7002", 1_000_000), (alpha_desk, "P-9002", 1_200_000), (beta_desk, "P-5002", 300_000)]
gate_rows = []
for desk, payment_id, amount in requests:
    proposal = Proposal(action="refund", payment_id=payment_id, amount=amount, reason="Fast Path에서 온 쓰기 제안")
    # 모델이 임의로 outcome을 executed(실행 완료)로 적어 내더라도 실제 실행 여부는 게이트가 결정합니다.
    answer = FinalAnswer(outcome="executed", text="환불했습니다.", proposal=proposal)
    gated = gate_answer(answer, desk, request_id="fast-demo")
    gate_rows.append({"고객": desk.scope.entity, "결제": payment_id, "제안의 outcome": answer.outcome, "게이트 판단": check_refund(proposal, desk.payments(), desk.scope.as_of).decision,
                      "최종 outcome": gated.outcome, "원장 건수": len(desk.ledger.refunds)})
show_table(gate_rows)

# hybrid 그래프의 간선 구조를 출력하여 fast와 writer가 모두 validate로 모이는지 확인합니다.
edges = build_hybrid(alpha_desk, model).get_graph().edges
show_table([{"출발": e.source, "도착": e.target, "조건부": e.conditional} for e in edges if e.source in ("fast", "writer", "validate")])
print("fast와 writer 노드는 모두 validate 노드로 모이며 동일한 게이트를 통과합니다. 일반 함수 호출이므로 모델 호출은 0회입니다.")

**관찰할 것.** Fast Path의 실질적인 이득은 오판 시 발생하는 복구 경로의 비용까지 합산해야 정확히 평가할 수 있습니다. 출력된 표에서 `Slow 전환`, `호출`, `입력 토큰`, `자동 검사`, `누락 사실` 컬럼을 함께 비교해 보세요.

- `Slow 전환`이 `True`인 행은 Fast 경로에서 먼저 소비한 호출과 Slow 경로로 되돌아가 다시 실행한 호출 비용을 **모두** 지불한 경우입니다. `역할별 호출`의 `fast` 항목이 복구 전에 허비한 호출 수이며, 이 행에서는 그만큼이 처음부터 Slow 경로로 갔을 때보다 추가로 든 비용이 됩니다. 따라서 Router의 Fast 오판이 잦아질수록 Fast Path의 평균 비용 절감 이득은 사라집니다.
- 반대로 `Slow 전환`이 `False`인데 `누락 사실`이 남아 있다면 훨씬 더 위험한 상황입니다. 코드 기반의 형식 검증은 인용 번호의 소속과 금액의 출처만 확인할 뿐, **조사 범위가 통째로 빠진 것**은 잡아내지 못합니다. 모델 호출과 토큰 사용량은 적고 답변 문장도 그럴듯해 보이므로, 비용 지표만 보면 마치 가장 훌륭한 실행처럼 착각하기 쉽습니다. (단, `누락 사실`이 `9월20일`처럼 날짜 표기 하나뿐이라면 답변 전문에 `2026-09-20` 같은 다른 형식으로 적혀 있는지 먼저 확인해 보세요. 단순 표기 차이라면 경로 설계의 위험이 아니라 문자열 일치 기반 자동 검사의 한계입니다.)
- 전환이 발생한 구체적인 이유는 `사건 순서` 출력의 `validate`와 `fallback_to_slow` 기록으로 확인할 수 있습니다. 현재의 전환 조건이 실제로 전환되어야 할 위험 사례들을 빠짐없이 잡아내는지 생각해 보세요. 사건 순서에 `gate`가 두 번 나타나는 행은 승인 대기(`needs_approval`)로 멈춘 경우입니다. 그래프 내부 상태에 게이트 처리 완료 표시가 남지 않아 외부 실행기가 동일한 요청 ID로 게이트를 한 번 더 호출하지만, 요청 ID가 같기 때문에 원장에 중복으로 기록되지는 않습니다. 전환 이후 `worker_checked`가 Worker마다 하나씩 나타나는 것은 합성 전에 코드가 각 Worker의 결과에서 유효하지 않은 인용을 걸러낸 기록입니다. 이벤트의 `dropped`는 등록되지 않은 번호여서 제거한 근거 참조 수이고, `recovered`는 번호 뒤에 부연 설명이 붙은 참조에서 유효한 번호만 추출해 살려낸 건수입니다. `filled`는 본문에 사실은 적혀 있으나 필터링 후 근거 참조가 하나도 남지 않은 결과에 대해 해당 Worker가 실제로 조회한 도구 결과에서 근거 참조를 보충해 넣은 건수입니다. 마지막으로 `gate_skipped`는 질문 속 결제 번호를 조회된 결제 내역과 대조할 수 없었음을 뜻하는 기록입니다(조회되지 않는 번호이거나 번호가 없을 때, 혹은 환불 요청 질문이거나 답변이 거절·심사 결론일 때 발생하며 답변 본문과 원장은 변경되지 않습니다).

게이트 검증 표를 보면 모델이 제출한 `제안의 outcome`은 모두 `executed`이지만, 실제 `최종 outcome`과 `원장 건수`는 코드 함수인 `게이트 판단`(`check_refund`)의 결과를 엄격히 따릅니다. 게이트 판단이 `allow`인 첫 번째 행만 실제로 실행되어 원장 건수가 1건이 되고, 나머지 행들은 거절되거나 승인 대기·심사 상태로 멈추어 원장 건수가 0건으로 유지됩니다. 즉, 쓰기 요청이 어느 경로를 통해 들어오든 반드시 `check_refund`를 거치게 됩니다. 아래쪽 간선 표에서 `fast`와 `writer` 노드가 모두 동일한 `validate` 노드로 모이는 것이 이를 보장하는 구조적 근거입니다. 이제 지금까지 개별적으로 살펴본 구조들을 동일한 조건의 비교표 하나로 모아 보겠습니다.

## 7. 같은 조건 비교표
**문제.** 앞선 절들에서 살펴본 수치들은 실행한 사례와 세부 조건이 조금씩 달라서 어떤 구조를 선택할지 결정하는 직접적인 근거로 삼기 어렵습니다. 공정한 비교표란 **비교하려는 대상(독립변수)만 하나로 줄이고 나머지 조건을 통제한 표**입니다. 여기서는 오직 에이전트 **구조**만 바꾸며, 사용하는 모델, 추론 강도(effort), 도구 집합, 코퍼스, 예산 한도, 기준일(`as_of`)은 모두 동일하게 고정합니다. 또한 모든 구조가 마지막에 동일한 쓰기 게이트(`gate_answer`)를 통과합니다.

수업 시간 안에 실행할 수 있도록 기본 비교 실험의 표본 크기는 작게 설정되어 있습니다. 따라서 이 비교표의 목적은 당장 어느 구조가 최종 승자인지 단정하는 것이 아니라, 무엇을 어떤 조건으로 측정해야 구조적 차이를 말할 수 있는지 그 방법을 익히는 것입니다.

**먼저 예측하세요.** 비교표를 실행하기 전에 사례 유형(`simple`, `complex`)별로 다섯 구조 중 어느 구조의 입력 토큰 소모가 가장 클지, 그리고 자동 검사 통과율이 떨어지는 조합이 있다면 어디일지 미리 예측해 적어 보세요.

**하네스가 비교의 일관성을 만듭니다.** 다섯 가지 구조(`single`, `single_skills`, `supervisor`, `handoff`, `hybrid`)는 내부의 에이전트 조립 방식만 다를 뿐, 모두 동일한 하네스(Harness) 위에서 동작합니다. 동일한 `Desk`의 예산(budget) 객체, 도구 허용 목록(`allowlist_middleware`), 최종 응답 형식(`FinalAnswer`), 그리고 쓰기 게이트(`gate_answer`)를 공통으로 거칩니다. 덕분에 비교표에 나타나는 수치 차이를 온전히 '구조의 차이'로 해석할 수 있습니다. 만약 구조마다 하네스 설정이 달랐다면, 결과의 차이가 구조 때문인지 하네스 때문인지 구분할 수 없었을 것입니다.

참고로 Deep Agents 프레임워크가 기본으로 자동 추가하는 `general-purpose` Worker는 우리의 하네스 통제 밖에서 동작합니다(02 노트북 3절 참고). 이 실습 과정에서는 `mas_lab` 패키지를 불러올 때 해당 기본 Worker가 생성되지 않도록 비활성화해 두었으므로, 다섯 구조 어디에서도 나타나지 않아야 합니다. 아래 코드 셀은 모델을 호출하지 않고 다섯 구조의 객체와 소스 코드를 직접 검사하여 이 조건들이 실제로 지켜지고 있는지 확인합니다. (`handoff`와 `hybrid`는 바깥쪽이 `StateGraph`이고 각 노드 내부에서 실행 시점에 에이전트가 생성되므로 그래프 자체에는 `task` 도구가 없습니다. 이 두 구조는 노드 내부와 동일한 설정으로 생성한 에이전트 객체를 통해 대신 확인합니다.)

### 그림으로 보기: 구조 다섯 개가 지나는 하네스

바로 아래 코드 셀에서 객체와 소스 코드로 검증할 내용입니다. 다섯 구조는 에이전트 조립 방식만 다르고 네 가지 핵심 하네스 장치는 모두 동일하게 공유합니다.

![single, single_skills, supervisor, handoff, hybrid 다섯 구조가 같은 하네스(예산, 도구 허용 목록, 응답 형식, 쓰기 게이트)를 지나 같은 칸의 실행 행이 된다](./assets/05b-3-same-harness.png)

- 하네스가 동일하게 통제되어야만 비교표의 수치 차이를 구조적 차이로 해석할 수 있습니다. 하네스가 구조마다 다르면 차이의 원인을 분리할 수 없습니다.
- 프레임워크가 기본으로 붙이는 `general-purpose` Worker는 하네스 통제를 벗어나므로 비활성화하여 제거합니다.

아래 코드는 다섯 구조를 생성하고 각각의 예산, 도구 허용 목록, 응답 형식, 쓰기 게이트, 그리고 자동 생성 Worker 제거 여부를 표로 점검합니다.

In [ ]:
# 모델을 호출하지 않고 다섯 구조를 생성하여 코드 객체와 소스의 하네스 일관성을 검사합니다.

# 1단계: 구조 이름별로 에이전트/그래프를 조립하는 함수와 소스 코드를 읽을 대상 함수를 매핑합니다.
BUILDERS = {
    "single": lambda desk: build_single(desk, model),
    "single_skills": lambda desk: build_single(desk, model, skills=True),
    "supervisor": lambda desk: build_supervisor(desk, model),
    "handoff": lambda desk: build_handoff(desk, model),
    "hybrid": lambda desk: build_hybrid(desk, model),
}
BUILDER_SOURCE = {"single": build_single, "single_skills": build_single, "supervisor": build_supervisor,
                  "handoff": build_handoff, "hybrid": build_hybrid}
assert set(BUILDERS) == set(STRUCTURES), "runner.STRUCTURES와 이 셀의 구조 목록이 일치하지 않습니다."

# 2단계: 각 구조의 역할별 도구 허용 목록을 코드 상수에서 가져와 정리합니다.
ALLOWED = {
    "single": ALL_TOOLS,
    "single_skills": ALL_TOOLS | SKILL_TOOLS,
    "supervisor": {"Supervisor": frozenset({"task"}) | ROLE_TOOLS["writer"], **ROLE_TOOLS},
    "handoff": OWNER_TOOLS,
    "hybrid": {"fast": FAST_TOOLS, **ROLE_TOOLS},
}
GRAPH_STRUCTURES = {"handoff", "hybrid"}  # 노드 내부에서 실행 시점에 에이전트가 만들어지는 그래프 구조

def allow_text(allowed):
    """도구 허용 목록을 보기 좋은 문자열로 포맷팅합니다."""
    if isinstance(allowed, dict):
        return "; ".join(f"{role}[{', '.join(sorted(tools))}]" for role, tools in allowed.items())
    return ", ".join(sorted(allowed))

# 3단계: 실행기(runner._run_turns)가 모든 구조의 최종 응답에 대해 쓰기 게이트(gate_answer)를 호출하는지 소스 코드로 확인합니다.
gate_in_runner = "gate_answer(" in inspect.getsource(runner._run_turns)

harness_rows = []
for name in STRUCTURES:
    # 4단계: 구조마다 새로운 Desk를 생성하여 조립만 수행합니다 (invoke를 호출하지 않으므로 비용이 들지 않습니다).
    harness_desk = make_desk(case_by_id["complex-double-pay"], label=f"harness/{name}")
    app = BUILDERS[name](harness_desk)
    source = inspect.getsource(BUILDER_SOURCE[name])

    # 5단계: 예산(Budget) 객체 공유 여부와 FIXED_BUDGET 한도 일치 여부를 확인합니다.
    same_limits = all(getattr(harness_desk.budget, key) == value for key, value in FIXED_BUDGET.items())
    uses_desk_budget = "desk.budget" in source or "checked_worker(" in source
    budget_text = "예 (desk.budget, 한도 FIXED_BUDGET)" if same_limits and uses_desk_budget else "아니오"

    # 6단계: 최종 응답 스키마(ProviderStrategy 내부 타입)를 소스에서 추출합니다.
    formats = sorted(set(re.findall(r"ProviderStrategy\((\w+)\)", source)))

    # 7단계: 쓰기 게이트 적용 여부를 표시합니다 (hybrid는 그래프 내부에도 gate 노드가 있습니다).
    gate_text = "예 (run_case의 gate_answer" + (" + gate 노드)" if name == "hybrid" else ")") if gate_in_runner else "아니오"

    # 8단계: 프레임워크의 기본 general-purpose Worker가 비활성화되었는지 task 도구의 담당자 목록으로 확인합니다.
    if name in GRAPH_STRUCTURES:
        probe = create_deep_agent(model=model, tools=harness_desk.tools("all"))  # 노드 내부와 동일한 기본 설정으로 확인
        targets, where = task_targets(probe), "같은 설정의 Agent"
    else:
        targets, where = task_targets(app), "구조 자체"
    auto_worker = "있음" if "general-purpose" in (targets or []) else "없음"

    harness_rows.append({"구조": name, "같은 Budget 객체를 쓰는가": budget_text, "도구 허용 목록": allow_text(ALLOWED[name]),
                         "최종 응답 형식": ", ".join(formats), "쓰기 게이트(gate_answer)": gate_text,
                         "자동 general-purpose Worker": f"{auto_worker} ({where}에서 확인, task 담당자 {targets})"})

show_table(harness_rows)
leaked = [row["구조"] for row in harness_rows if row["자동 general-purpose Worker"].startswith("있음")]
assert not leaked, f"자동 general-purpose Worker가 남아 있는 구조: {leaked}"
print("모델 호출 0회")

**2026년 11월 주요 프레임워크 공식 문서가 기본으로 제공하는 멀티에이전트 구성과 이 노트북의 구조 비교**

> 실무에서 어느 멀티에이전트 구성이 가장 높은 비율로 사용되는지 엄밀하게 측정한 공식 조사는 확인되지 않았습니다. 출처가 불분명한 "70%가 특정 패턴을 쓴다"와 같은 수치는 인용하지 않습니다. 대신 아래 표는 사용 빈도가 아니라 **2026-11-02 기준 주요 프레임워크 공식 문서들이 기본 구성으로 안내하고 있는 패턴**을 이 노트북의 다섯 구조와 짝지은 것입니다.

| 이 노트북의 구조 | 다룬 절 | 주요 프레임워크 공식 문서의 대응 구성 |
|---|---|---|
| Supervisor (`build_supervisor`) | 02 3절 | LangChain Subagents, OpenAI Agents as tools (manager), Microsoft Magentic (manager agent), Google ADK Collaborative workflow (coordinator가 sub-agent를 호출), Anthropic orchestrator-worker (lead agent와 병렬 subagent) |
| Handoff (`build_handoff`) | 02 4절 | LangChain Handoffs, OpenAI Handoffs, Microsoft Handoff |
| Router + Workflow (`build_hybrid`) | 03b 4절(Router), 04b 8절(조립) | LangChain Router 및 Custom workflow, OpenAI code orchestration (structured output 기반 routing, chaining, evaluator loop, parallel), Microsoft Sequential 및 Concurrent, Google ADK Template workflows (sequence, loop, parallel) 및 Routing |
| Skills (`single_skills`) | 05a 4절 | LangChain Skills |
| 이 과정에서 다루지 않음 | | Microsoft Group Chat |

이 표를 읽는 핵심 포인트는 세 가지입니다.

- 프레임워크마다 부르는 이름은 달라도 실제 작동 구조가 같은 경우가 많습니다. **"누가 다음 행동을 결정하는가"**와 **"실행 결과가 호출자에게 되돌아오는가"**를 기준으로 보면 manager, orchestrator, coordinator는 모두 Supervisor 패턴에 해당하고, Handoff는 대화의 주도권(담당자) 자체를 넘기는 패턴에 해당합니다.
- 코드가 실행 순서를 직접 통제하는 구성(Sequential, Concurrent, Template workflows)은 이 노트북에서 Workflow 영역, 즉 `hybrid`의 그래프 구조에 해당합니다. 05a 0절에서 코드가 잘할 수 있는 영역을 굳이 에이전트로 나누지 않았던 것과 동일한 설계 판단입니다.
- 이 표는 각 프레임워크 구성의 **개념적 이름**을 짝지은 것일 뿐, 어느 쪽의 성능이 더 우수하다고 말하지 않습니다. 우리 업무에 어느 구조가 더 적합한지는 바로 이어지는 동일 조건 비교표를 통해 직접 측정해 판단해야 합니다.

참고 출처 (2026-11-02 기준 확인된 공식 문서 및 기술 블로그):

- LangChain Multi-agent: https://docs.langchain.com/oss/python/langchain/multi-agent
- OpenAI Agents SDK, Multi-agent orchestration: https://openai.github.io/openai-agents-python/multi_agent/
- Microsoft Agent Framework, Workflow orchestrations (문서 갱신일 2026-08-25): https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations
- Google ADK, Workflows: https://adk.dev/workflows/
- Anthropic, How we built our multi-agent research system (2025-06-13): https://www.anthropic.com/engineering/multi-agent-research-system

아래 코드는 동일 조건 비교 실험에 사용할 통제 조건과 대상 사례 목록, 그리고 예상 호출 상한을 출력합니다.

In [ ]:
# 비교 실험에 사용할 세 가지 구조와 두 가지 사례 유형, 반복 횟수를 설정합니다.
CONFIG = {"STRUCTURES": ["single", "supervisor", "hybrid"], "CASE_TYPES": ["simple", "complex"], "REPEATS": 1}
grid_cases = [case for case in cases if case["type"] in CONFIG["CASE_TYPES"]]
runs = len(CONFIG["STRUCTURES"]) * len(grid_cases) * CONFIG["REPEATS"]

# 앞서 실행한 복합 사례의 호출 수를 기준으로 전체 격자 실험의 상한을 어림잡아 확인합니다.
measured = {"single": single_double["model_calls"], "supervisor": supervisor_row["model_calls"], "hybrid": hybrid_rows[1]["model_calls"]}
worst = sum(measured[name] for name in CONFIG["STRUCTURES"]) * len(grid_cases) * CONFIG["REPEATS"]

show_table([{"고정 조건": key, "값": value} for key, value in {
    "모델": settings.model, "effort": settings.effort, "SDK 재시도": "없음(max_retries=0)", "도구": sorted(ALL_TOOLS), "검색": "BM25", "예산": FIXED_BUDGET,
    "쓰기 경로": "gate_answer", "바꾸는 변수": "구조"}.items()])
show_table([{key: case[key] for key in ("id", "type", "entity", "as_of", "question")} for case in grid_cases])
print(f"{len(CONFIG['STRUCTURES'])}구조 × {len(grid_cases)}사례 × {CONFIG['REPEATS']}회 = {runs}행")
print(f"앞서 측정한 복합 사례 호출 수로 모든 행을 어림잡은 상한: 모델 약 {worst}회 {measured}. 단순 사례는 이보다 적게 호출됩니다. 모델 호출 0회.")

In [ ]:
# 세 가지 구조와 대상 사례들에 대해 동일 조건 비교 실험(run_grid)을 실행합니다.
announce("같은 조건 비교표 실행", model_calls=100)
grid_rows = run_grid(CONFIG["STRUCTURES"], grid_cases, model=model, repeats=CONFIG["REPEATS"])
print(f"{len(grid_rows)}행 실행 완료. 실패한 행도 삭제하지 않고 그대로 남깁니다.")

In [ ]:
# 실행된 모든 행에 대해 정답 데이터(gold) 기반의 자동 검사를 수행하고 전체 행 표를 출력합니다.
grid_checked = [evaluate_row(row, gold) for row in grid_rows]
grid_table = [{"행": index, "구조": row["structure"], "유형": row["type"], "사례": row["case_id"], "상태": row["status"], "outcome": row["outcome"],
               "자동 검사": check["자동 검사"], "실패 칸": check["실패 칸"], "누락 사실": check["누락 사실"], "금지 포함": check["금지 포함"],
               "호출": row["model_calls"], "도구": row["tool_calls"], "입력 토큰": row["input_tokens"], "시간": row["elapsed_seconds"], "겹침": row["max_worker_overlap_seconds"],
               "최종 응답자": row["final_speaker"], "원장": row["ledger_refunds"], "오류": row["errors"][:1]} for index, (row, check) in enumerate(zip(grid_rows, grid_checked, strict=True))]
show_table(grid_table)
print(f"전체 {len(grid_table)}행 (삭제한 행 없음). 자동 검사 실패 {sum(not row['자동 검사'] for row in grid_table)}행, 실행 실패 {sum(row['상태'] != 'ok' for row in grid_table)}행. 모델 호출 0회.")

In [ ]:
# 구조별·사례 유형별로 실행 지표와 통과 수를 요약한 표를 출력합니다.
grid_summary = summarize_rows(grid_rows, grid_checked)
show_table(grid_summary, columns=["구조", "유형", "실행", "자동 검사 통과", "실행 실패", "평균 시간", "평균 모델 호출", "평균 입력 토큰", "usage 누락", "정책 위반"])
print(f"각 칸의 표본은 {CONFIG['REPEATS']}회 반복, 유형당 {len(grid_cases) // len(CONFIG['CASE_TYPES'])}사례입니다. 반복 횟수가 3회 미만이므로 통계적 우열을 주장하지 않습니다. 모델 호출 0회.")

In [ ]:
# 앞 절에서 실행한 행과 방금 비교표에서 실행한 행(동일 구조, 동일 사례, 동일 조건)을 나란히 놓아 자연 변동폭을 확인합니다.
twins = []
for name, earlier in (("single", single_rows[0]), ("supervisor", supervisor_row), ("hybrid", hybrid_rows[1])):
    later = next(row for row in grid_rows if row["structure"] == name and row["case_id"] == "complex-double-pay")
    for label, row in (("앞 절 실행", earlier), ("비교표 실행", later)):
        twins.append({"구조": name, "실행": label, "호출": row["model_calls"], "도구": row["tool_calls"], "입력 토큰": row["input_tokens"], "시간": row["elapsed_seconds"],
                      "자동 검사": evaluate_row(row, gold)["자동 검사"]})
show_table(twins)
print("동일한 구조, 동일한 사례, 동일한 조건을 두 번 측정한 결과입니다. 추가 API 호출 없이 이미 실행된 두 행을 나란히 비교했습니다. 모델 호출 0회.")

In [ ]:
# 선택 실습: 4개 유형 전체와 5개 구조 전체를 비교하는 확장 실험입니다 (기본값은 False).
if not RUN_OPTIONAL:
    print("실행하지 않았습니다. 전체 비교를 실행하려면 첫 번째 코드 셀의 RUN_OPTIONAL을 True로 바꾸어 실행해 보세요. 모델 호출 약 220~330회, 10~15분 규모입니다.")
else:
    announce("전체 비교 4유형 × 5구조 × 1회 실행", model_calls=330)
    full_cases = [case for case in cases if case["type"] in ("simple", "complex", "insufficient", "approval")]
    full_rows = run_grid(list(STRUCTURES), full_cases, model=model)
    full_checked = [evaluate_row(row, gold) for row in full_rows]
    show_table(summarize_rows(full_rows, full_checked))

In [ ]:
# 실패 재현: 평균치만 보면 잘못 해석하기 쉬운 표를 재현하기 위해, 예산을 낮추어 일찍 종료되는 실패 행을 일부러 하나 만듭니다.
announce("예산을 낮춘 실패 행 1개 생성", model_calls=3)
cut_row = run_case("single", double_pay, model=model, budget={**FIXED_BUDGET, "max_model_calls": 2})
same = [pair for pair in zip(grid_rows, grid_checked, strict=True) if pair[0]["type"] == "complex"]
# 기존 복합 사례 결과에 예산을 낮춰 일찍 실패한 행(cut_row)과 앞서 Fast Path를 강제한 행(forced_rows[0])을 섞습니다.
mixed = [*same, (cut_row, evaluate_row(cut_row, gold)), (forced_rows[0], evaluate_row(forced_rows[0], gold))]
names = CONFIG["STRUCTURES"]

def means(pairs):
    """주어진 실행 행들의 평균 시간, 평균 호출 수, 평균 입력 토큰, 자동 검사 통과 수를 계산합니다."""
    count, tokens = len(pairs), [row["input_tokens"] for row, _ in pairs if row["input_tokens"] is not None]
    average = lambda key: round(sum(row[key] for row, _ in pairs) / count, 1) if count else None
    return {"행": count, "평균 시간": average("elapsed_seconds"), "평균 호출": average("model_calls"), "평균 입력 토큰": round(sum(tokens) / len(tokens)) if tokens else None,
            "자동 검사 통과": sum(check["자동 검사"] for _, check in pairs), "남은 사례": sorted({row["case_id"] for row, _ in pairs})}

def table(pairs, name_filter=lambda pair: True):
    """조건 필터(name_filter)에 따라 구조별 요약 평균 표를 생성합니다."""
    return [{"구조": name, **means([p for p in pairs if p[0]["structure"] == name and name_filter(p)]),
             "제외한 행": sum(p[0]["structure"] == name and not name_filter(p) for p in pairs)} for name in names]

print("평균 0. 같은 조건의 복합 사례 (실패 행 포함)"); show_table(table(same))
print("평균 A. 평균 0에 일찍 끝난 실패 행 2개를 섞은 경우 (실패 행 포함)"); show_table(table(mixed))
print("평균 B. 평균 A에서 자동 검사 실패 행을 제외한 경우 (실패 행 제외)"); show_table(table(mixed, lambda pair: pair[1]["자동 검사"]))
print("실패 행 목록 (평균 표 계산 방식과 무관하게 삭제하지 않고 함께 제시합니다):")
show_table([{"구조": row["structure"], "사례": row["case_id"], "상태": row["status"], "호출": row["model_calls"], "입력 토큰": row["input_tokens"], "시간": row["elapsed_seconds"],
             "누락 사실": check["누락 사실"], "실패 칸": check["실패 칸"]} for row, check in mixed if not check["자동 검사"]])

In [ ]:
# 자동 검사에서 실패한 행(없으면 첫 번째 복합 사례 행)을 골라 사람이 직접 원문과 대조해 봅니다.
failed_rows = [row for row, check in zip(grid_rows, grid_checked, strict=True) if not check["자동 검사"]]
review_row = failed_rows[0] if failed_rows else next(row for row in grid_rows if row["case_id"] == "complex-double-pay")
print(f"[답] 구조={review_row['structure']} 사례={review_row['case_id']} outcome={review_row['outcome']}\n{review_row['answer']}\n")
print("[자동 검사]", {key: value for key, value in evaluate_row(review_row, gold).items() if key in ("자동 검사", "실패 칸", "누락 사실", "금지 포함")})
print("[인용]", review_row["evidence_refs"])

# 인용된 문서와 필수 정답 문서의 원문 중 숫자가 포함된 핵심 행들을 함께 출력합니다.
documents = build_retriever().documents
for doc_id in sorted({*review_row["cited_docs"], *gold[review_row["case_id"]]["required_docs"]}):
    print(f"\n[원문] {documents[doc_id].source}")
    for number, line in enumerate(documents[doc_id].text.split("\n"), start=1):
        if line.strip() and any(ch.isdigit() for ch in line):
            print(f"  {number}: {line}")

# 답변의 문장과 원문 행을 직접 대조하여 자동 검사 결과와 다르게 판단되는 지점을 적어 보세요.
HUMAN_REVIEW = {"대조한 문장": "미판정", "자동 검사와 다른 판단": "미판정", "근거 원문 행": "미판정"}
print("\nHUMAN_REVIEW:", HUMAN_REVIEW, "| 모델 호출 0회")

**관찰할 것.** 전체 행 표, 구조별 요약 표, 같은 구조로 두 번 측정한 표, 세 가지 평균 비교표, 그리고 사람 판정 칸을 차례대로 살펴보세요. **이 비교표는 어느 구조가 절대적으로 우월한지 단정하지 않습니다.**

**1회 관찰과 자연 변동.** 요약 표의 각 칸은 사례당 1회씩만 실행한 결과입니다. 여러 번 반복 측정하지 않았기 때문에 구조 자체의 차이와 실행할 때마다 발생하는 우연한 변동을 구분할 수 없습니다. '같은 구조로 두 번 측정한 표'(`twins`)를 확인해 보세요. 동일한 구조와 동일한 사례인데도 `호출`, `입력 토큰`, `자동 검사` 결과가 얼마나 달라졌나요? 만약 서로 다른 두 구조 사이의 수치 차이가 이 자연 변동폭보다 작다면, 그 차이를 구조 덕분이라고 주장할 수 없습니다. 현재 표로 말할 수 있는 정확한 사실은 "이번 1회 실행에서 이 조합의 행이 이러한 수치를 기록했다"까지입니다.

**평균의 함정.** 세 가지 평균 표(평균 0, 평균 A, 평균 B)를 순서대로 비교해 보세요.

- **평균 0에서 평균 A로의 변화.** 사례 구성은 그대로 둔 채, 예산 한도에 걸려 일찍 멈춘 실패 행과 Fast 강제 행만 추가했습니다. 만약 `평균 시간`과 `평균 입력 토큰`이 줄어들었다면, **중도 실패가 평균 비용을 오히려 좋아 보이게 착시 효과를 만든 것**입니다. 반드시 `행` 수 대비 `자동 검사 통과` 수가 어떻게 변했는지 함께 확인해야 합니다. 통과율을 가린 채 평균 시간만 제시하면 실패한 구조가 마치 비용을 절감한 것처럼 잘못 읽힙니다.
- **평균 A에서 평균 B로의 변화.** 자동 검사에 실패한 행을 표에서 제외하면 각 구조에 `남은 사례` 목록이 서로 달라질 수 있습니다. 까다로운 사례가 실패해 빠지고 쉬운 사례만 남으면 평균 비용이 내려가고, 반대로 쉬운 사례가 빠지고 어려운 사례만 남으면 평균 비용이 올라갑니다. 또한 일찍 멈춘 실패 행이 제외되면 평균 시간이 다시 늘어나기도 하므로, 수치의 증감 방향보다 **`남은 사례` 구성이 구조마다 달라졌는지**를 먼저 확인해야 합니다. 남아 있는 사례가 서로 다르면 구조 간 공정한 비교 자체가 성립하지 않습니다. 만약 `남은 사례`가 평균 0과 똑같아졌다면 평균 B는 평균 0으로 되돌아간 것일 뿐이며, 제외한 실패 행이 왜 발생했는지는 표에서 감춰진 상태가 됩니다. 따라서 `제외한 행`이 존재하는 구조의 평균 B 수치는 액면 그대로 비교해서는 안 됩니다.
- **실패 행 목록 확인.** `상태`가 `ok`인데 실패 행 목록에 오른 경우는 에이전트 실행은 정상 종료되었지만 자동 검사 기준을 통과하지 못한 행입니다. `누락 사실`과 `실패 칸`을 실제 답변과 대조해 보세요. 핵심 사실이 진짜로 누락된 경우와 내용은 맞지만 표현 형식만 다른 경우는 구분해서 다루어야 합니다. 만약 `누락 사실`은 비어 있는데 `실패 칸`에 `outcome`이나 `인용 문서`가 적혀 있다면, 답변 텍스트에 키워드는 들어 있지만 최종 처리 상태나 근거 문서 연결이 어긋난 경우입니다. 전체 행 표의 `outcome`과 답변 본문을 읽고 원인을 구분해 보세요. `outcome`만 어긋났다면 단순 문의에 환불 제안을 낸 것처럼 결과 유형이 잘못된 것이고, `insufficient`로 끝나면서 `인용 문서`까지 어긋났다면 하위 Worker의 조사 결과가 근거 참조(`evidence_refs`) 없이 상위로 전달되었을 가능성을 점검해 보아야 합니다. 이처럼 요약 평균 표는 반드시 실패 행 목록과 함께 읽어야 합니다.

**usage 누락 확인.** 요약 표의 `usage 누락`이 0이 아니라면, 토큰 사용량이 보고되지 않은 호출이 있었다는 뜻이므로 그만큼 평균 토큰 수치가 실제보다 낮게 집계될 수 있습니다(누락된 호출은 0으로 처리하지 않고 평균 분모에서 제외해 계산합니다).

**자동 검사의 한계와 사람의 원문 대조.** 마지막 셀의 사람 판정 출력은 자동 검사가 실패한 행을 우선 보여 줍니다(실패 행이 없으면 복합 사례 행을 출력합니다). 출력된 답변 문장과 아래쪽 `[원문]` 행들을 나란히 대조해 보세요. 문자열 포함 여부만 보는 자동 검사는 표현이 조금만 달라도 정답을 오답으로 판정하고, 반대로 부정문에 키워드가 들어 있어도 정답으로 통과시키는 한계가 있습니다. 또한 자동 검사를 통과한 행이더라도 인용된 문서가 답변의 문장을 실제로 뒷받침하는지 확인해 보세요. 환불 금액은 맞췄지만 엉뚱한 조항을 인용했을 수도 있습니다. 직접 읽어 본 판단이 자동 검사 결과와 다르다면 `HUMAN_REVIEW` 딕셔너리에 기록해 보세요. 전체 행 표의 `자동 검사` 결과를 사람의 판정으로 바로잡아 읽으면 요약 표의 인상이 어떻게 달라지는지도 함께 살펴보세요.

**운영 결론을 내리려면 무엇을 더 측정해야 할까요?** 반복 횟수를 늘려 실행 간 변동폭을 확인하고, 사례의 수와 유형을 넓히며, 사람이 직접 원문과 대조해 판정한 지표를 추가해야 합니다. 설령 그렇게 하더라도 그 결론은 이 실습의 코퍼스와 모델 조건 안에서만 유효합니다. 실제 서비스 운영에 필요한 Router의 오판 비율, 실트래픽의 단순/복합 질문 분포, API 호출당 단가 등은 이 표에 포함되어 있지 않으므로, 이 표 하나만으로 실무 운영에서 특정 구조가 무조건 더 낫다고 단정할 수 없다는 점을 기억해 두세요.

## 8. 선택 기록
**문제.** 에이전트 개수나 화려한 조직도 그림은 설계의 최종 결과물이 아닙니다. 나중에 다른 동료나 미래의 내가 이 시스템의 구조를 유지할지 바꿀지 판단할 때 진짜 필요한 것은 **"왜 이 구조를 선택했고, 그 대가로 무엇을 포기했으며, 어떤 측정 결과가 나오면 이 결정을 재검토할 것인가"**를 적은 설계 이유입니다.

구조를 선택할 때 답해야 할 핵심 설계 질문은 여섯 가지입니다.

1. **업무 목표.** 어떤 상태를 작업 완료로 정의할 것인가요? (01 노트북)
2. **Agent·SubAgent의 필요성.** 일반 코드 함수나 Skill만으로 충분하지 않은 이유는 무엇인가요? (02 노트북, 05a 4절)
3. **작업 Graph.** 순차, 병렬, 분기, 루프 흐름은 각각 어디에 배치되나요? (03a, 03b 노트북)
4. **Context.** 각 작업 담당자가 볼 수 있는 정보와 격리해야 할 정보는 무엇인가요? (04a 노트북 3절)
5. **권한과 예산, 종료.** 누가 어떤 도구를 쓸 수 있고, 언제 실행을 멈추나요? (04a 노트북 4절, 04b 노트북 5절)
6. **평가 근거.** 어떤 측정 결과로 이 구조 선택을 정당화하나요? (이 노트북 7절)

### 그림으로 보기: 흩어진 선택 기준을 묻는 순서

위 여섯 질문 가운데 2번(Agent·SubAgent의 필요성)에 답하는 기준은 05a와 05b의 여러 절에 흩어져 있습니다. 기록을 적기 전에 그 기준을 묻는 순서대로 한 장에 모았습니다. 왼쪽 질문부터 답하고, "예"이면 그 아래 칸에서 멈춥니다. 칸 아래의 숫자는 그 기준을 다룬 절입니다.

![선택 순서도. 코드로 정해지나, Single로 고쳐지나, 격리나 병렬이 필요한가, 주도권이 넘어가나를 차례로 묻고, 예이면 코드 노드, Single과 Skill, Supervisor나 Send, Handoff에서 멈춘다. 모두 아니오이면 나누지 않는다. 어느 칸이든 돈을 움직이는 쓰기는 gate_answer 하나를 지난다](./assets/05b-4-decision-flow.png)

- **코드로 정해지나?** 판단이 필요 없는 칸(범위 확인, 정책 판정, 계산, 쓰기)은 Agent 후보가 아닙니다. 05a 0절의 담당 표에서 `담당`이 코드인 행입니다. 이 칸을 Agent 여러 개로 쪼개면 새로 생기는 판단은 없고 조정 비용만 늡니다.
- **Single로 고쳐지나?** 05a 1절에서 실패를 먼저 분류하고 프롬프트, 도구 설명, 검색어를 고쳐 봅니다. 같은 절차를 매번 설명하기 싫은 것뿐이라면 Skill(05a 4절)로 충분합니다. 여기서 고쳐진 문제를 Agent를 늘려 고치면 같은 품질을 더 비싸게 사는 것입니다.
- **격리·병렬이 필요한가?** 서로 무관한 원문이 한 Context에 쌓여 판단을 흐리거나, 역할마다 쓸 수 있는 도구를 달리 막아야 하거나, 서로 기다릴 필요 없는 조사가 지연을 키울 때입니다(05a 2~3절, 05b 5절). 이 칸에서 멈췄다면 다음으로 **작업을 누가 나누는가**를 묻습니다. 분해를 코드가 미리 정할 수 있으면 `Send` 병렬(03a 3절)이 더 싸고 실행마다 같은 모양으로 돕니다. 입력을 보고 그때그때 나눠야 하면 부모 모델이 위임하는 Supervisor(02 3절)를 씁니다. Worker가 늘어 중간 계층이 따로 걸러 낼 판단이 생기면 Hierarchical을 검토하지만, 걸러 낼 판단이 없는 계층은 지연과 전달 손실만 더합니다(02 5절).
- **주도권이 넘어가나?** 고객과의 대화 자체를 다른 담당자(owner)가 이어받아야 할 때입니다(02 4절). 이 질문은 바로 앞 질문과 다른 축이어서 둘 다 "예"일 수 있습니다. 담당자를 넘긴 뒤 그 담당자 안에서 병렬 조사를 할 수 있기 때문입니다(02 6절).
- **모두 아니오이면** 아직 나눌 근거가 없습니다. Single로 두고, 무엇을 더 재야 판단할 수 있는지를 아래 선택 기록에 적습니다.

**어느 칸에서 끝나든 바뀌지 않는 것이 하나 있습니다.** 돈을 움직이는 쓰기는 구조와 상관없이 코드 게이트 하나(`gate_answer`)를 지납니다. 이 과정이 처음부터 이렇게 설계한 이유는 바깥에서 나온 실무 보고와도 맞습니다.

- Anthropic은 자사의 멀티에이전트 리서치 시스템을 소개하면서(2025-06-13) Agent는 채팅보다 토큰을 약 4배, 멀티에이전트 시스템은 약 15배 쓴다고 보고했습니다. 그래서 병렬화할 여지가 크고 그 비용을 치를 만큼 값어치 있는 작업에 맞으며, 모든 Agent가 같은 Context를 공유해야 하거나 Agent 사이에 의존이 많은 영역은 아직 잘 맞지 않는다고 적었습니다. 코딩도 리서치보다 진짜로 병렬화할 수 있는 작업이 적다고 했습니다.
- Cognition은 「Don't Build Multi-Agents」(2025-06-12)에서 두 원칙을 내세웠습니다. Context를 공유하되 메시지 하나가 아니라 Agent의 전체 실행 기록을 공유하라는 것, 그리고 행동에는 겉으로 드러나지 않은 결정이 담겨 있어서 서로 어긋난 결정은 나쁜 결과를 낸다는 것입니다.
- LangChain은 이 두 글을 나란히 놓고(2025-06-16) "읽기 행동은 쓰기 행동보다 본래 병렬화하기 쉽다"고 정리했습니다. 서로 어긋난 쓰기는 서로 어긋난 읽기보다 훨씬 나쁜 결과를 내기 때문입니다. 같은 글은 Anthropic의 시스템도 조사는 여러 Agent가 나누어 하지만 최종 보고서는 주 Agent 하나가 쓴다고 짚습니다.

세 글은 같은 쪽을 가리킵니다. **나누기 쉬운 것은 읽기이고, 쓰기와 결정은 한 곳에 모읍니다.** 이 과정에서 조사 Worker는 여럿으로 나누되 결제 원장 쓰기는 `gate_answer` 하나로 모은 것이 그 예이고, 다음 06의 공개 웹 리서치도 이렇게 나누기 쉬운 읽기 작업입니다. 다만 세 글의 수치와 판단은 각 회사의 작업과 모델에서 나온 것입니다. 우리 업무에서는 7절처럼 같은 조건으로 다시 재어 확인합니다.

- https://www.anthropic.com/engineering/multi-agent-research-system
- https://cognition.com/blog/dont-build-multi-agents
- https://www.langchain.com/blog/how-and-when-to-build-multi-agent-systems

아래 코드 셀의 `DECISION` 딕셔너리에 사례 유형(`simple`, `complex`)별로 선택한 구조, 그 근거가 되는 7절 전체 행 표의 `행` 번호, 그리고 그 선택으로 포기한 요소를 적어 보세요. 만약 1회 실행 결과만으로는 결정을 내릴 수 없다고 판단된다면 `"미판정"`으로 남겨 두어도 좋습니다. 1회 관찰만으로 단정할 수 없다는 사실과 앞으로 무엇을 더 측정해야 하는지를 명시하는 것 또한 훌륭한 엔지니어링 기록입니다.

In [ ]:
# 사례 유형별로 선택한 구조, 근거 행 번호, 포기한 요소를 기록합니다.
DECISION = {"simple": {"구조": "미판정", "근거 행": [], "포기한 것": "미판정"},
            "complex": {"구조": "미판정", "근거 행": [], "포기한 것": "미판정"}}
# 작성 예시: DECISION["simple"] = {"구조": "single", "근거 행": [0, 1], "포기한 것": "Worker를 분리해 얻는 원문 Context 격리"}

show_table([{"유형": kind, **entry, "근거 행 내용": [(grid_table[i]["구조"], grid_table[i]["사례"], grid_table[i]["자동 검사"]) for i in entry["근거 행"]]}
            for kind, entry in DECISION.items()])
print("근거 행 번호는 7절 전체 행 표(grid_table)의 '행' 컬럼 번호입니다. 모델 호출 0회.")

## 직접 해보기

유료 API를 호출하는 실습 셀은 첫 줄의 `RUN_EXERCISE = False`를 `True`로 변경해야 실제로 실행됩니다. 여러분이 직접 수정해 볼 설정값들은 그 바로 아래에 대문자 상수로 모아 두었습니다. 모델을 호출하지 않는 일반 코드 셀은 수정 없이 바로 실행됩니다. (반복 실행에 따른 변동폭 확인과 예산 한도 축소 실습은 05a 노트북에 포함되어 있습니다.)

### 해보기 1

(실제 모델 호출) `billing-policy` 문서의 규정 표현을 일상적인 질문으로 바꾼 새로운 평가 사례를 직접 추가해 봅니다. 정답으로 기대하는 사실(`required_any`)은 반드시 **원문 행**의 표현에 근거해 연결해야 합니다. 아래 코드 셀은 기대 표현이 포함된 원문 행을 먼저 찾아 출력합니다. 원본 사례 파일을 수정하지 않고 이 셀의 변수만으로 안전하게 실행하고 평가합니다. 값을 수정한 뒤 `RUN_EXERCISE`를 `True`로 바꾸어 실행해 보세요.

In [ ]:
RUN_EXERCISE = False
# 질문 표현과 기대 정답 조건을 원문 행에 맞추어 정의합니다.
NEW_CASE = {"id": "my-refund-window", "type": "simple", "entity": "알파", "as_of": "2026-09-08", "question": "결제하고 석 달쯤 지나면 환불은 어떻게 처리되나요?"}
NEW_EXPECT = {"outcome": "answered", "required_any": [["90일"], ["재무 담당", "심사"]], "required_docs": ["billing-policy"], "forbidden": [], "ledger_refunds": 0, "route": "fast"}

# billing-policy 원문에서 '90일'이 포함된 행을 먼저 찾아 확인합니다.
policy_text = build_retriever().documents["billing-policy"].text.split("\n")
print("기대 사실의 원문 행:", [(number, line) for number, line in enumerate(policy_text, start=1) if "90일" in line])

if not RUN_EXERCISE:
    print("실행하지 않았습니다. 값을 수정한 뒤 RUN_EXERCISE를 True로 바꾸어 실행해 보세요.")
else:
    announce("새 사례 1건 실행", model_calls=15)
    new_row = run_case("single", NEW_CASE, model=model)
    show_table([evaluate_row(new_row, {**gold, NEW_CASE["id"]: NEW_EXPECT})], columns=["case_id", "자동 검사", "실제 outcome", "누락 사실", "누락 문서", "실패 칸"])
    print(new_row["answer"])

### 해보기 2

문자열 포함 여부만 검사하는 `required_any` 자동 검사에서 **내용은 맞지만 오답으로 판정되는 답변(위음성)**, 그리고 반대로 **내용은 완전히 틀렸지만 정답으로 통과하는 답변(위양성)**을 직접 작성해 보고 그 이유를 확인합니다. 모델을 호출하지 않으므로 비용이 들지 않습니다. 아래 코드에 들어 있는 두 예시 문장을 확인한 뒤 여러분만의 문장으로 바꾸어 실행해 보세요.

In [ ]:
# 모델을 호출하지 않고 수동으로 작성한 답변 문자열에 대해 자동 검사(evaluate_row)의 한계를 테스트합니다.
BASE = {"structure": "손으로 쓴 답", "case_id": "complex-double-pay", "status": "ok", "outcome": "needs_approval", "ledger_refunds": 0,
        "cited_docs": ["billing-policy", "sla-a-v2", "sla-a-amendment"]}

# 내용은 정확하지만 '150,000원'을 '15만 원'으로, '9월 20일'을 '2026-09-20'으로 표기하여 자동 검사에 실패하는 예시
CORRECT_BUT_FAILS = "P-9002 1,200,000원을 승인 후 환불합니다. 가용률 99.3%로 크레딧은 기본 이용료의 15%, 15만 원이고 2026-09-20까지 신청하세요."
# 핵심 숫자와 키워드를 모두 포함하고 있지만 문맥은 정반대(환불 및 크레딧 거절)인데도 자동 검사를 통과하는 예시
WRONG_BUT_PASSES = "P-9002는 환불 대상이 아닙니다. 1,200,000원은 그대로 둡니다. 99.3%라도 15%를 받지 못합니다. 150,000원은 드리지 않으며 9월 20일은 무관합니다."

show_table([{"답": answer, **{key: check[key] for key in ("자동 검사", "누락 사실", "금지 포함")}}
            for answer in (CORRECT_BUT_FAILS, WRONG_BUT_PASSES) for check in [evaluate_row({**BASE, "answer": answer}, gold)]])
print("표현 형식이 조금만 달라도 옳은 답을 놓치고, 문맥을 이해하지 못하므로 의미가 정반대여도 키워드만 있으면 통과합니다. 모델 호출 0회.")

In [ ]:
# 오늘 작성한 평가 근거와 구조 선택 기록을 설계 로그 파일에 저장합니다.
EVALUATION_BASIS = "미작성: 어떤 결과로 구조 선택을 정당화할지, 무엇을 더 측정해야 하는지 적어 보세요."
answer = json.dumps({"근거": EVALUATION_BASIS, "선택": DECISION, "사람 판정": HUMAN_REVIEW}, ensure_ascii=False)
path = write_design_log("평가 근거: 어떤 결과로 선택을 정당화할까?", answer)
print("기록 저장 위치:", path.relative_to(ROOT))
print("지금까지 기록된 설계 질문:", list(read_design_log()))

## 정리

앞서 학습한 05a의 정리 내용과 이어서 전체 흐름을 정리해 보세요.

- **모든 수치는 네 가지 조건(비교 대상, 평가 방법, 표본, 측정자)과 함께 제시합니다.** 전체 지연 시간은 에이전트의 개수가 아니라 가장 긴 실행 경로(Critical Path)를 기준으로 해석합니다.
- **Fast Path는 안전 검사를 생략하는 경로가 아닙니다.** Fast Path의 진짜 이득은 라우팅 오판 확률과 Slow 전환 복구 비용까지 합산하여 평가해야 하며, 누락된 조사 범위는 단순 형식 검증만으로 잡아내지 못할 수 있습니다.
- **실패한 실행 행을 임의로 삭제하지 않습니다.** 예산 초과 등으로 일찍 멈춘 실패 행이 섞이면 평균 시간과 비용이 오히려 낮아져 좋아 보이는 착시가 발생하므로, 항상 통과율 및 실패 행 목록과 함께 읽어야 합니다.
- **단 1회 관찰한 결과만으로 구조의 우열을 단정하지 않습니다.** 현재 실험으로 말할 수 있는 것과 아직 말할 수 없는 것(추가 측정이 필요한 항목)을 구분해 기록하는 것까지가 비교표 작성의 완성입니다.

참고로 05a에서 작성한 `1절 진단`(`DIAGNOSIS`) 변수는 이 노트북이 05a와 독립된 커널에서 실행되기 때문에 위 설계 로그 저장 시 자동으로 합쳐지지 않습니다(파일에는 이전 노트북에서 저장한 항목들이 그대로 유지됩니다).

## 확인 질문

1. 비교 대상(사례 구성 등)이 서로 다른 두 수치를 하나로 묶어 비율을 말하면 왜 오류가 되나요? 방금 실행한 5절의 표에서 구체적인 예를 들어 설명해 보세요.
2. 평균 지연 시간과 토큰 소모가 낮아졌는데도 실제로는 시스템 품질이 나빠진 경우는 언제인가요? 7절의 평균 0, 평균 A, 그리고 실패 행 목록을 근거로 설명해 보세요.
3. 7절의 동일 조건 비교표만으로는 아직 주장할 수 없는 내용을 세 가지 이상 적어 보세요.

**다음 노트북 안내.** 이어지는 **06** 노트북에서는 EXA 웹 검색 도구를 활용해 병렬(`parallel`), 직렬(`serial`), 연쇄(`chained`) 리서치 Deep Agent를 직접 조립하고 동일한 연구 주제로 비교 실험을 수행합니다. 사내 고객 요청이 아닌 공개 웹 리서치 문제에 이 노트북의 측정과 판단 방식을 그대로 적용해 봅니다.